# exp_003 — Context × quantization interaction

This notebook regenerates measured outputs only. It does not run inference or invent missing cells.

The interaction label is descriptive: `approximately_constant`, `context_dependent`, or `insufficient_data`.

In [13]:
import os
from pathlib import Path
import runpy
import sys
import pandas as pd
import matplotlib.pyplot as plt

ROOT = next(candidate for candidate in [Path.cwd(), *Path.cwd().parents] if (candidate / 'src').is_dir() and (candidate / 'experiments').is_dir())
EXPERIMENT_ROOT = ROOT / 'experiments/exp_003-context_x_quantization'
sys.path.insert(0, str(ROOT / 'src'))
from llm_lab.analysis import (
    aggregate_jsonl,  # validation is centralized by analyze.py
    effective_context_by_variant_and_task,
    interaction_report,
    matched_cell_rows,
    relative_degradation_rows,
)

# Change PHASE to 'pilot' or 'main' for a measured phase.
# PHASE = 'smoke' is permitted only for explicit harness validation.
PHASE = os.environ.get('EXP003_PHASE', 'smoke')
RESULTS_DIR = Path(os.environ.get('EXP003_RESULTS_DIR', ROOT / 'experiments/exp_003-context_x_quantization/results'))
ALLOW_FIXTURE = os.environ.get('EXP003_ALLOW_FIXTURE', '0') == '1'
if ALLOW_FIXTURE and PHASE != 'smoke':
    raise ValueError('EXP003_ALLOW_FIXTURE=1 is permitted only for PHASE=smoke')
RAW_PATH = RESULTS_DIR / 'raw' / f'{PHASE}-trials.jsonl'
SUMMARY_PATH = RESULTS_DIR / 'processed' / 'summary.csv'
MANIFEST_PATH = RESULTS_DIR / 'manifests' / f'{PHASE}.json'
FINDINGS_PATH = ROOT / 'docs/findings.md'

analysis_ready = MANIFEST_PATH.is_file()
if not analysis_ready:
    print(f'exp_003 {PHASE} results are not available: {MANIFEST_PATH}')
    print('Run the exp_003 runner first, then rerun this notebook.')
else:
    spec_path = EXPERIMENT_ROOT / 'analyze.py'
    namespace = runpy.run_path(
        str(spec_path),
        run_name='exp_003_analysis',
    )
    regenerate = namespace['regenerate']
    regeneration = regenerate(
        MANIFEST_PATH,
        allow_fixture=ALLOW_FIXTURE,
    )
    if not SUMMARY_PATH.is_file():
        raise FileNotFoundError(f'processed summary is required: {SUMMARY_PATH}')
    summary_rows = regeneration['summary_rows']
    available_summary_rows = pd.read_csv(SUMMARY_PATH)
    available_summary_rows.head()

AnalysisInputError: exp_003 requires scorer version 'calibrated.v1'

In [8]:
if analysis_ready:
    run_manifest = __import__('json').loads(MANIFEST_PATH.read_text(encoding='utf-8'))
    variant_ids = tuple(item['condition_id'] for item in run_manifest['quantization_variants'])
    context_lengths = tuple(run_manifest['context_lengths'])
    evidence_positions = tuple(run_manifest['evidence_positions'])
    task_types = tuple(run_manifest['task_types'])
    task_ids = tuple(run_manifest['task_ids'])
    matched = matched_cell_rows(summary_rows, variant_ids=variant_ids, context_lengths=context_lengths, evidence_positions=evidence_positions, task_types=task_types, task_ids=task_ids)
    degradation = relative_degradation_rows(matched, baseline_context_tokens=min(context_lengths))
    gap_reports = interaction_report(matched, reference_variant=run_manifest['analysis']['primary_gap_reference'], approx_constant_gap_tolerance=run_manifest['analysis']['approx_constant_gap_tolerance'])
    gap_report_frame = pd.DataFrame(gap_reports)
    gap_report_frame[['task_type', 'variant_condition_id', 'classification', 'gap_change', 'matched_n']]

In [9]:
if analysis_ready:
    FIGURES_DIR = RESULTS_DIR / 'figures'
    FIGURES_DIR.mkdir(parents=True, exist_ok=True)
    matched_frame = pd.DataFrame(matched)

    # context × quantization accuracy heatmaps by task type.
    for task_type in task_types:
        plot_frame = matched_frame[matched_frame['task_type'] == task_type]
        heatmap = plot_frame.groupby(['target_context_tokens', 'variant_condition_id'], as_index=False)['accuracy'].mean().pivot(index='target_context_tokens', columns='variant_condition_id', values='accuracy')
        figure, axis = plt.subplots(figsize=(7, 4))
        image = axis.imshow(heatmap.to_numpy(dtype=float), vmin=0, vmax=1, aspect='auto', cmap='viridis')
        axis.set_title(f'Context × quantization accuracy — {task_type}')
        axis.set_xlabel('Quantization variant')
        axis.set_ylabel('Context tokens')
        axis.set_xticks(range(len(heatmap.columns)), heatmap.columns)
        axis.set_yticks(range(len(heatmap.index)), heatmap.index)
        figure.colorbar(image, ax=axis, label='Accuracy')
        figure.tight_layout()
        figure.savefig(FIGURES_DIR / f'context-x-quantization-{task_type}.png', dpi=160)
        plt.show()

In [10]:
if analysis_ready:
    # position × context accuracy heatmaps for each quantization.
    for variant_id in variant_ids:
        plot_frame = matched_frame[matched_frame['variant_condition_id'] == variant_id]
        heatmap = plot_frame.groupby(['target_context_tokens', 'requested_evidence_position'], as_index=False)['accuracy'].mean().pivot(index='target_context_tokens', columns='requested_evidence_position', values='accuracy')
        figure, axis = plt.subplots(figsize=(7, 4))
        image = axis.imshow(heatmap.to_numpy(dtype=float), vmin=0, vmax=1, aspect='auto', cmap='viridis')
        axis.set_title(f'Position × context accuracy — {variant_id}')
        axis.set_xlabel('Requested evidence position')
        axis.set_ylabel('Context tokens')
        axis.set_xticks(range(len(heatmap.columns)), [f'{value:.0%}' for value in heatmap.columns])
        axis.set_yticks(range(len(heatmap.index)), heatmap.index)
        figure.colorbar(image, ax=axis, label='Accuracy')
        figure.tight_layout()
        figure.savefig(FIGURES_DIR / f'position-x-context-{variant_id}.png', dpi=160)
        plt.show()

In [11]:
if analysis_ready:
    # Direct quantization_gap versus context, retaining matched scored counts.
    # analyze.py writes relative-degradation.csv and interaction.json.
    gap_points = [dict(task_type=report['task_type'], variant_condition_id=report['variant_condition_id'], **point) for report in gap_reports for point in report['context_points']]
    gap_points_frame = pd.DataFrame(gap_points)
    figure, axis = plt.subplots(figsize=(8, 4))
    for (task_type, variant_id), group in gap_points_frame.groupby(['task_type', 'variant_condition_id']):
        axis.plot(group['context_tokens'], group['quantization_gap'], marker='o', label=f'{task_type} / {variant_id}')
    axis.set_title('Matched quantization gap versus context')
    axis.set_xlabel('Context tokens')
    axis.set_ylabel('Reference accuracy − variant accuracy')
    axis.legend()
    figure.tight_layout()
    figure.savefig(FIGURES_DIR / 'quantization-gap-vs-context.png', dpi=160)
    plt.show()
    gap_report_frame[['task_type', 'variant_condition_id', 'classification', 'shortest_context_gap', 'largest_context_gap', 'gap_change']]

In [12]:
if analysis_ready:
    # The run manifest records scorer_version and may classify a gap as insufficient_data.
    effective = effective_context_by_variant_and_task(matched, baseline_context_tokens=min(context_lengths), alpha=run_manifest['effective_context']['alpha'], minimum_baseline_accuracy=run_manifest['effective_context']['baseline_accuracy_gate'])
    effective_frame = pd.DataFrame(effective)
    effective_frame[['variant_condition_id', 'task_type', 'status', 'effective_context_tokens', 'crossing_context_tokens']]

    # accuracy_degradation is relative to each variant's short-context baseline.
    pd.DataFrame(degradation)[['task_id', 'variant_condition_id', 'target_context_tokens', 'accuracy_degradation', 'relative_degradation']]

    if 'exp_003: not yet measured' in FINDINGS_PATH.read_text(encoding='utf-8'):
        print('exp_003 is not yet measured; no finding is added from this notebook run.')
    else:
        print('Only reviewed real-model data may update docs/findings.md.')